# ClaimClerk agent — UC function tools + managed MCP

Builds the ClaimClerk agent: UC function tools + a managed Vector Search MCP server.

## What this does

Wires the ClaimClerk agent on top of three tool sources:

1. Three **Unity Catalog functions** — data-join lookups over the
   PawShield Delta tables: `get_customer_policy(customer_id)` (reads
   `customers` + `policies` + `pets`), `get_claim_history(customer_id,
   policy_year)` (reads `claims` joined to `policies`), and
   `check_in_network(vet_id)` (reads `vets`). The agent calls these
   for the caller's policy context, prior-claim history, and
   vet-network status respectively.
2. A **managed Vector Search MCP server** pointing at
   `policy_chunks_index`. The agent calls this
   when it needs the actual policy-clause text.
3. A **Python callable** `flag_for_human_review(...)` for
   escalation. The agent invokes this when it lacks confidence
   to answer directly.

The agent is wrapped in MLflow's `ResponsesAgent` envelope so it
can be registered to UC and deployed via Model Serving.

## Prerequisites

* `customers`, `policies`, `claims` Delta tables
  populated.
* `policy_chunks_index` Vector Search index exists
  (verify with the listing below before running the agent cell).
  If the workspace's VS endpoint has been torn down, run the
  c0801-build-policy-index.py builder notebook first to re-provision.
* `pip install databricks-mcp databricks-langchain langgraph "mlflow[databricks]"`
  (these are the packages the Agent Framework docs reference;
  pinned versions land in `requirements.txt` if you log the
  model).
* Serverless compute (Databricks default).

## Source verification

* **ResponsesAgent** wrapper pattern — verified May 2026 against
  the [agent-authoring docs](https://docs.databricks.com/aws/en/generative-ai/agent-framework/author-agent).
* **`DatabricksMCPClient`** + managed-MCP URL patterns — verified
  May 2026 against the [managed MCP docs](https://docs.databricks.com/aws/en/generative-ai/mcp/managed-mcp).
* **UC functions as agent tools** — UC `CREATE FUNCTION` syntax.

In [0]:
# Pin mlflow<3.13 — 3.13.0 has a regression where eval_item.trace is None
# during mlflow.genai.evaluate (trace not associated with eval_request_id).
%pip install --quiet --force-reinstall databricks-mcp databricks-langchain langgraph "mlflow[databricks]>=3.12,<3.13" nest_asyncio

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jupyter-server 1.23.4 requires anyio<4,>=3.1.0, but you have anyio 4.13.0 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("catalog", "genaicert")
dbutils.widgets.text("vs_endpoint", "pawshield-vs")
dbutils.widgets.text("index_name", "policy_chunks_index")
# Source: https://docs.databricks.com/aws/en/machine-learning/foundation-model-apis/supported-models
# `databricks-meta-llama-3-3-70b-instruct` is the documented pay-per-token
# endpoint name and supports function/tool calling (verified May 2026).
# 70B is required for the multi-tool ReAct loop here (UC function +
# managed MCP + escalation tool); the 8B endpoint frequently skips the
# retrieval step and escalates on turn 1, which the tightened prompt
# below also forbids.
dbutils.widgets.text("llm_endpoint", "databricks-meta-llama-3-3-70b-instruct")
catalog = dbutils.widgets.get("catalog")
vs_endpoint = dbutils.widgets.get("vs_endpoint")
index_short = dbutils.widgets.get("index_name")
llm_endpoint = dbutils.widgets.get("llm_endpoint")
print(f"Catalog: {catalog}  |  VS endpoint: {vs_endpoint}  |  Index: {index_short}")
print(f"LLM endpoint: {llm_endpoint}")

Catalog: genaicert  |  VS endpoint: pawshield-vs  |  Index: policy_chunks_index
LLM endpoint: databricks-meta-llama-3-3-70b-instruct


## 1. Verify the prerequisites are in place

In [0]:
%sql
-- Customers + policies tables should be populated by the setup notebook.
-- We need Sarah Chen's row for the demo query.
-- Schema note (May 2026): customers has first_name/last_name (no
-- full_name column) and address_state (no plain `state` column).
SELECT
  customer_id,
  CONCAT_WS(' ', first_name, last_name) AS full_name,
  address_state                         AS state
FROM IDENTIFIER(:catalog || '.pawshield.customers')
WHERE customer_id = 'CUST-CHEN-001';

customer_id,full_name,state
CUST-CHEN-001,Sarah Chen,TX


In [0]:
# Vector Search index should exist (built by c0801).
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient(disable_notice=True)
index_name = f"{catalog}.pawshield.{index_short}"
try:
    info = vsc.get_index(endpoint_name=vs_endpoint, index_name=index_name).describe()
    print(f"Found {index_name}  status={info.get('status', {}).get('ready')}")
except Exception as e:
    print(
        f"Could not describe {index_name}: {type(e).__name__}: {e}\n"
        "Run the c0801-build-policy-index.py notebook first, or override the `vs_endpoint` and "
        "`index_name` widgets to match your workspace's actual names."
    )

Found genaicert.pawshield.policy_chunks_index  status=True


## 2. Define `get_customer_policy` as a UC function

UC functions are first-class tools for Mosaic AI Agent Framework
— the agent discovers them by name and binds them automatically
when wrapped in the right adapter. We define a small one that
joins customers + policies + pets so the agent gets the full
context in a single tool call.

**SQL vs Python form.** Two documented registration paths exist:
SQL `CREATE FUNCTION` (used here) and the Python
`DatabricksFunctionClient().create_python_function(...)` API.
Pick by what the tool *is*: a data-join over Delta tables is
naturally SQL — the function body is itself a SELECT, so SQL DDL
keeps the lookup logic in one language. For tools whose logic is
pure-Python (date math, string normalisation, an API client call
with no Spark), the Python form is the documented preferred path
because the client introspects the callable's type hints +
docstring instead of re-stating them in DDL. Both surfaces
produce a registered UC function the agent discovers identically.

In [0]:
%sql
-- Column-name notes (verified against the live UC schema, May 2026):
--   policies.annual_deductible  (aliased to `deductible` for the agent)
--   policies.terminated_at      (NULL => active; surfaced as `active` BOOLEAN)
--   customers.address_state     (aliased to `state` for the agent)
--   pets.name                   (aliased to `pet_name` for the agent)
-- The tool-side names are stable for the LLM; the underlying schema names
-- can rename without breaking the agent's prompt.
CREATE OR REPLACE FUNCTION IDENTIFIER(:catalog || '.pawshield.get_customer_policy')(customer_id STRING)
RETURNS TABLE (
  policy_id   STRING,
  tier        STRING,
  state       STRING,
  pet_name    STRING,
  deductible  DOUBLE,
  active      BOOLEAN
)
COMMENT 'Look up the active policies for a PawShield customer by customer_id. Returns one row per active policy, joining customer state and pet name.'
RETURN
  SELECT
    p.policy_id,
    p.tier,
    c.address_state              AS state,
    pet.name                     AS pet_name,
    p.annual_deductible          AS deductible,
    (p.terminated_at IS NULL)    AS active
  FROM IDENTIFIER(:catalog || '.pawshield.policies') p
  JOIN IDENTIFIER(:catalog || '.pawshield.customers') c
    ON p.customer_id = c.customer_id
  LEFT JOIN IDENTIFIER(:catalog || '.pawshield.pets') pet
    ON p.pet_id = pet.pet_id
  WHERE p.customer_id = get_customer_policy.customer_id
    AND p.terminated_at IS NULL;

In [0]:
# Smoke-test the function against Sarah's customer_id.
result = spark.sql(
    f"SELECT * FROM {catalog}.pawshield.get_customer_policy('CUST-CHEN-001')"
).toPandas()
print(result)
# Expected: 2 rows (Sarah owns Mochi the Shiba and Biscuit the cat, both
# on PawPlus tier per the generator seed pawshield_v1).

             policy_id     tier state pet_name  deductible  active
0  POL-2024-08-CHENS01  PawPlus    TX    Mochi       250.0    True
1  POL-2025-01-CHENB02  PawPlus    TX  Biscuit       250.0    True


## 2b. Two more UC-function tools — `get_claim_history` + `check_in_network`

Reasoning over Sarah's appeal needs more than her policy: the agent
has to read her **claim history** for the policy year (to discover
the January claim already met the $250 deductible) and, for
vet-network questions, whether a practice is **in network**. Both
are data-join lookups over Delta tables, so they register as UC
functions exactly like `get_customer_policy`.

Two design points the chapter makes concrete here:

* **Return-shape discipline.** `get_claim_history` returns only the
  fields the agent reasons over — claim id, date, status,
  billed/reimbursed, the policy deductible, and a derived
  `deductible_applied` flag — not the full claim row. A tool that
  dumped every column would burn context budget the agent needs for
  the rest of the turn.
* **Docstring scopes selection.** `check_in_network` answers
  *participation only*; its COMMENT says so explicitly, so the LLM
  doesn't reach for it on an availability question (live scheduling
  comes from an External MCP server, not from this UC function).

In [0]:
%sql
-- claims carries policy_id (not customer_id), so get_claim_history
-- joins claims -> policies to filter by customer_id, and filters by
-- policy year (year of visit_date). There is no deductible_applied
-- column in the data; we DERIVE it: TRUE when the reimbursement
-- withheld more than the plain copay share implies, i.e. a deductible
-- was taken on that claim. Seeing it TRUE on more than one claim in
-- the same policy year is how the agent spots the $250 deductible
-- re-applied on Sarah's March claim (the billing error she appeals).
CREATE OR REPLACE FUNCTION IDENTIFIER(:catalog || '.pawshield.get_claim_history')(
    customer_id STRING,
    policy_year INT
)
RETURNS TABLE (
  claim_id           STRING,
  visit_date         DATE,
  status             STRING,
  amount_billed      DOUBLE,
  amount_reimbursed  DOUBLE,
  annual_deductible  DOUBLE,
  deductible_applied BOOLEAN
)
COMMENT 'Return a PawShield customer''s claim history for one policy year (by visit_date year). One row per claim across all the customer''s policies, with the policy annual deductible and a derived flag for whether a deductible was withheld on that claim. Use this when the question is about prior claims, claim status, reimbursement amounts, or whether the annual deductible was already met this year. Do NOT use it for policy coverage terms (use get_customer_policy) or vet-network status (use check_in_network).'
RETURN
  SELECT
    cl.claim_id,
    cl.visit_date,
    cl.status,
    cl.amount_billed,
    cl.amount_reimbursed,
    p.annual_deductible,
    -- copay_pct is the reimbursement rate (e.g. 0.80 = 80% reimbursed); a
    -- payout below billed*rate signals the annual deductible reduced it.
    (cl.amount_reimbursed IS NOT NULL
       AND cl.amount_reimbursed < round(cl.amount_billed * p.copay_pct, 2) - 0.01) AS deductible_applied
  FROM IDENTIFIER(:catalog || '.pawshield.claims') cl
  JOIN IDENTIFIER(:catalog || '.pawshield.policies') p
    ON cl.policy_id = p.policy_id
  WHERE p.customer_id = get_claim_history.customer_id
    AND year(cl.visit_date) = get_claim_history.policy_year
  ORDER BY cl.visit_date;

In [0]:
# Smoke-test get_claim_history against Sarah's 2026 claims.
result = spark.sql(
    f"SELECT * FROM {catalog}.pawshield.get_claim_history('CUST-CHEN-001', 2026)"
).toPandas()
print(result)
# Expected: the two canonical 2026 appeal claims for Sarah —
# CLM-2026-01-00098 (Jan, approved, 420 -> 136) and
# CLM-2026-03-00471 (Mar, partial_reimbursement, 890 -> 512) — both with
# deductible_applied = True, plus any bulk-generated 2026 claims the seed
# added on Sarah's policy.
# More than one deductible-applied claim in the same policy year is the
# signal the agent reasons over: the $250 annual deductible applies once.

            claim_id  visit_date  ... annual_deductible  deductible_applied
0  CLM-2026-01-00098  2026-01-22  ...             250.0                True
1  CLM-2026-03-00471  2026-03-14  ...             250.0                True
2  CLM-2026-04-05906  2026-04-10  ...             250.0                True

[3 rows x 7 columns]


In [0]:
%sql
-- check_in_network is a single-table lookup over the vets directory.
-- It returns participation status only — NOT live appointment
-- availability (that lives in the external scheduling system wired
-- via an External MCP server).
CREATE OR REPLACE FUNCTION IDENTIFIER(:catalog || '.pawshield.check_in_network')(
    vet_id STRING
)
RETURNS TABLE (
  vet_id        STRING,
  practice_name STRING,
  city          STRING,
  state         STRING,
  in_network    BOOLEAN
)
COMMENT 'Look up whether a veterinary practice participates in PawShield''s network, by vet_id. Returns practice name, location, and in_network status. Use ONLY for network participation / eligibility questions. It does NOT return appointment availability or open scheduling slots — live availability comes from the external scheduling MCP server.'
RETURN
  SELECT
    v.vet_id,
    v.practice_name,
    v.address_city  AS city,
    v.address_state AS state,
    v.in_network
  FROM IDENTIFIER(:catalog || '.pawshield.vets') v
  WHERE v.vet_id = check_in_network.vet_id;

In [0]:
# Smoke-test check_in_network against the vet that treated Biscuit.
result = spark.sql(
    f"SELECT * FROM {catalog}.pawshield.check_in_network('VET-AUSTIN-0042')"
).toPandas()
print(result)
# Expected: 1 row — Lone Star Animal ER (Austin, TX), in_network = True.

            vet_id        practice_name    city state  in_network
0  VET-AUSTIN-0042  Lone Star Animal ER  Austin    TX        True


## 3. Configure the managed Vector Search MCP server

The managed MCP server URL format (verified May 2026 against the
docs) is:

`https://<workspace>/api/2.0/mcp/vector-search/{catalog}/{schema}/{index_name}`

The `DatabricksMCPClient` handles auth via the workspace SDK.

In [0]:
# Databricks notebooks run inside a running asyncio event loop, and
# `DatabricksMCPClient.list_tools()` / `.call_tool()` internally call
# `asyncio.run(...)` — which fails with "asyncio.run() cannot be called
# from a running event loop". `nest_asyncio.apply()` patches asyncio to
# allow nested run() calls, which is the documented workaround for
# running asyncio-internals inside a notebook host loop.
import nest_asyncio
nest_asyncio.apply()

In [0]:
from databricks.sdk import WorkspaceClient
from databricks_mcp import DatabricksMCPClient

workspace = WorkspaceClient()

vs_mcp_url = (
    f"{workspace.config.host}"
    f"/api/2.0/mcp/vector-search/{catalog}/pawshield/{index_short}"
)
print(f"Managed MCP URL: {vs_mcp_url}")

vs_mcp_client = DatabricksMCPClient(
    server_url=vs_mcp_url,
    workspace_client=workspace,
)

Managed MCP URL: https://<workspace>.cloud.databricks.com/api/2.0/mcp/vector-search/genaicert/pawshield/policy_chunks_index


In [0]:
# Source: https://docs.databricks.com/aws/en/generative-ai/mcp/managed-mcp
# `list_tools()` returns `mcp.types.Tool` Pydantic objects (with `.name`,
# `.description`, `.inputSchema`). These are NOT LangChain `BaseTool` objects,
# so `create_react_agent(tools=...)` can't consume them directly. The
# documented adapter pattern is to wrap each MCP tool with `@tool` from
# `langchain_core.tools`, with a closure over `vs_mcp_client.call_tool(...)`
# for execution.
vs_mcp_raw_tools = vs_mcp_client.list_tools()
print(f"Tools exposed by VS MCP server: {[t.name for t in vs_mcp_raw_tools]}")

Tools exposed by VS MCP server: ['genaicert__pawshield__policy_chunks_index']


In [0]:
from typing import Any, Optional

from langchain_core.tools import StructuredTool
from pydantic import BaseModel, ConfigDict, create_model

# MCP inputSchema → Pydantic field-type mapping. Flat-schema only; if a
# managed MCP tool ever exposes a nested object/array schema, extend here.
_JSON_PY_TYPES: dict[str, type] = {
    "string":  str,
    "integer": int,
    "number":  float,
    "boolean": bool,
    "array":   list,
    "object":  dict,
}

def _args_schema_from_mcp(tool_name: str, input_schema: dict) -> type[BaseModel]:
    """Build a Pydantic model from an MCP tool's `inputSchema`, with
    `additionalProperties=false` (Pydantic `extra='forbid'`).
    The Databricks chat-FM tool-use API rejects schemas that allow
    `additionalProperties` (returns a BAD_REQUEST indicating the
    JSON schema is invalid because the `additionalProperties`
    keyword must be False or not specified),
    so we forbid extra fields explicitly when generating the args
    schema for `StructuredTool`."""
    properties = (input_schema or {}).get("properties", {}) or {}
    required = set((input_schema or {}).get("required", []) or [])
    fields: dict[str, Any] = {}
    for prop_name, prop_schema in properties.items():
        py_type = _JSON_PY_TYPES.get(prop_schema.get("type", "string"), str)
        if prop_name in required:
            fields[prop_name] = (py_type, ...)
        else:
            fields[prop_name] = (Optional[py_type], None)
    return create_model(
        f"{tool_name}Args",
        __config__=ConfigDict(extra="forbid"),
        **fields,
    )


def _wrap_mcp_tool(client, mcp_tool):
    """Adapt one MCP-protocol Tool into a LangChain `StructuredTool` by
    closing over the MCP client's `call_tool` method. The wrapper
    inherits the MCP tool's name + description so the LLM can select it
    normally, and uses an explicit Pydantic `args_schema` (with
    `extra='forbid'`) so the schema Databricks sees has no
    `additionalProperties`."""
    name = mcp_tool.name
    description = mcp_tool.description or f"Managed MCP tool: {name}"
    args_schema = _args_schema_from_mcp(name, getattr(mcp_tool, "inputSchema", {}))

    def _run(**kwargs):
        # MCP `call_tool` returns a CallToolResult with `.content` (list of
        # TextContent blocks). Stringify so LangGraph treats it as a tool
        # observation it can pass back to the LLM.
        result = client.call_tool(name, kwargs)
        if hasattr(result, "content") and result.content:
            return "\n".join(getattr(c, "text", str(c)) for c in result.content)
        return str(result)

    return StructuredTool.from_function(
        func=_run,
        name=name,
        description=description,
        args_schema=args_schema,
    )

In [0]:
vs_mcp_tools = [_wrap_mcp_tool(vs_mcp_client, t) for t in vs_mcp_raw_tools]
print(f"Wrapped {len(vs_mcp_tools)} MCP tool(s) as LangChain StructuredTool adapters.")

Wrapped 1 MCP tool(s) as LangChain StructuredTool adapters.


## 4. Define a Python-callable escalation tool

In [0]:
from langchain_core.tools import tool

@tool
def flag_for_human_review(reason: str, customer_id: str) -> str:
    """Escalate a request to a human reviewer when the agent lacks
    confidence to answer. Use when the question requires policy
    interpretation outside the standard clauses, when emotional/legal
    sensitivity is high, or when the retrieved policy text contradicts
    the question's premise."""
    # In production this writes to a review-queue Delta table. For the
    # notebook demo, we return a deterministic acknowledgement so the
    # agent's behavior is observable in the trace.
    return (
        f"Escalated to human reviewer. Customer: {customer_id}. "
        f"Reason: {reason}"
    )

## 5. Build the agent with `create_react_agent` + ResponsesAgent

The pattern (per current docs):

1. Build the agent with any supported framework (here:
   LangGraph's `create_react_agent` with `ChatDatabricks` as the
   LLM client).
2. Bind tools — Python callables, MCP tools, UC-function tools.
3. Wrap in `ResponsesAgent` for deployment compatibility.

In [0]:
# Enable MLflow autolog so the agent's internal tool calls and LLM calls
# are captured as child spans of the predict trace. create_react_agent
# is a LangGraph runnable, and
# mlflow.langchain.autolog() traces LangChain AND LangGraph calls — so the
# trace shows one span per tool call (get_customer_policy,
# get_claim_history, the VS retrieval tool, the LLM) instead of only the
# outer predict span.
import mlflow
mlflow.langchain.autolog()

In [0]:
# Patch: databricks-vectorsearch 0.74 renamed VectorSearchIndex → AISearchIndex,
# but databricks-ai-bridge 0.19.0 still imports the old name at package init.
import databricks.vector_search.client as _vsc_mod
if not hasattr(_vsc_mod, 'VectorSearchIndex'):
    _vsc_mod.VectorSearchIndex = _vsc_mod.AISearchIndex

from databricks_langchain import ChatDatabricks
# Source: https://docs.databricks.com/aws/en/generative-ai/agent-framework/create-custom-tool
# Canonical import is from the package root, not the `uc_ai` submodule (the
# submodule path was an older/internal layout that the public docs no longer use).
from databricks_langchain import UCFunctionToolkit
from langgraph.prebuilt import create_react_agent

# (1) UC function tools — three data-join lookups over Delta tables
uc_toolkit = UCFunctionToolkit(
    function_names=[
        f"{catalog}.pawshield.get_customer_policy",
        f"{catalog}.pawshield.get_claim_history",
        f"{catalog}.pawshield.check_in_network",
    ]
)
uc_tools = uc_toolkit.tools

# (2) MCP tools already collected as `vs_mcp_tools`
# (3) Python-callable: flag_for_human_review

all_tools = [*uc_tools, *vs_mcp_tools, flag_for_human_review]

/local_disk0/.ephemeral_nfs/envs/pythonEnv-b2431b4d-1564-4f48-9446-b1c7e96109f6/lib/python3.10/site-packages/databricks/connect/session.py:451: UserWarning: Ignoring the default notebook Spark session and creating a new Spark Connect session. To use the default notebook Spark session, use DatabricksSession.builder.getOrCreate() with no additional parameters.
  warnings.warn(new_notebook_session_msg)


In [0]:
print(f"Total tools bound: {len(all_tools)}")
for t in all_tools:
    # Every tool here is now a LangChain BaseTool (UC functions and the MCP
    # adapters above both produce BaseTools; flag_for_human_review uses the
    # @tool decorator). Read `.name` directly.
    print(f"  - {t.name}")

Total tools bound: 5
  - genaicert__pawshield__get_customer_policy
  - genaicert__pawshield__get_claim_history
  - genaicert__pawshield__check_in_network
  - genaicert__pawshield__policy_chunks_index
  - flag_for_human_review


In [0]:
# Pedagogy note: prompt rules guide an LLM; they do not bind it. The
# May 2026 trace audit caught three failure modes — 8B escalating on
# turn 1, 70B retrieving but escalating without citing a Section, and
# escalation reasons that didn't name what was searched. The fixes
# below stack four reinforcements: citation pulled to RULE 1 (most
# prominent), self-check instruction ("verify Section X.Y appears
# before responding"), Sarah-Chen-specific few-shot with the exact
# canonical answer, and a "WRONG vs RIGHT" pair showing the audit's
# observed failure mode against the desired shape. The harder lesson:
# prompt rules are necessary but insufficient; the
# production-ready pattern adds a programmatic output validator that
# rejects non-compliant responses and triggers retry.
SYSTEM_PROMPT = (
    "You are ClaimClerk, PawShield's claims-reasoning assistant.\n"
    "\n"
    "RULE 1 (CITATION) — Every final customer-facing answer MUST "
    "contain a 'Section X.Y' citation drawn from a policy chunk you "
    "retrieved this turn. No citation = no answer. If you cannot "
    "cite, you have not retrieved enough; retrieve again.\n"
    "\n"
    "RULE 2 (NO PREMATURE ESCALATION) — NEVER call "
    "flag_for_human_review until you have completed BOTH "
    "get_customer_policy AND at least one VS retrieval. First-turn "
    "escalation is forbidden.\n"
    "\n"
    "RULE 3 (ESCALATION FORMAT) — If you do escalate, the reason "
    "field MUST name (a) the query terms you searched, (b) which "
    "retrieved sections you considered, and (c) why they didn't "
    "address the question. 'Insufficient information' alone is "
    "non-compliant.\n"
    "\n"
    "WORKFLOW — execute in order:\n"
    "1. Call get_customer_policy(customer_id) to fetch the caller's "
    "active policies (policy_id, tier, state, annual_deductible).\n"
    "2. If the question is about a claim, reimbursement, or the "
    "deductible, call get_claim_history(customer_id, policy_year) "
    "for the relevant year. If deductible_applied is true on more "
    "than one claim in the same policy year, the annual deductible "
    "was charged more than once — it applies only ONCE per policy "
    "year, so flag the recurrence in your reasoning.\n"
    "3. Call the VS retrieval tool (managed MCP) with the customer's "
    "specific question terms, filters {state, tier} from step 1. "
    "Query for the MOST SPECIFIC governing clause: a deductible-"
    "recurrence question is governed by the chronic-condition "
    "exclusion clause, not the general reimbursement-formula clause; "
    "cite the most specific retrieved clause, not the general one.\n"
    "4. Reason over the policy clause and the claim history together. "
    "If the retrieved chunks don't address the question, retry with "
    "refined terms before giving up.\n"
    "5. Compose the answer. Before sending, SELF-CHECK: does the "
    "draft contain 'Section X.Y' where X.Y is a real section from a "
    "retrieved chunk? If no, you have failed RULE 1 — retrieve more "
    "or rewrite citing the chunk you retrieved.\n"
    "\n"
    "FEW-SHOT — Sarah Chen's appeal (canonical scenario):\n"
    "  User: 'My reimbursement was only $512 on an $890 bill in "
    "March, but in January it was $136 on a $420 bill. Why the "
    "difference?'\n"
    "  Turn 1: get_customer_policy(customer_id='CUST-CHEN-001')\n"
    "  Turn 2: get_claim_history(customer_id='CUST-CHEN-001', "
    "policy_year=2026) -> CLM-2026-01-00098 (Jan, approved) and "
    "CLM-2026-03-00471 (Mar, partial) BOTH show "
    "deductible_applied=true, so the already-satisfied $250 annual "
    "deductible was wrongly re-applied on the March claim, leaving "
    "that reimbursement $200 short of the $712 owed ($512 actual).\n"
    "  Turn 3: VS retrieve query='chronic condition exclusion "
    "deductible once per policy year' filters {state: 'TX', tier: "
    "'PawPlus'}\n"
    "  Turn 4: 'Per Section 4.7 of policy POL-2025-01-CHENB02, the "
    "$250 annual deductible applies once per policy year and resets "
    "at renewal. Your January claim ($420 billed) had the $250 "
    "deductible applied, then 80%% coinsurance on the remaining $170 "
    "= $136 reimbursed — correct. By March the deductible was already "
    "satisfied, so it should not apply again: 80%% of the $890 bill = "
    "$712 owed. You were reimbursed only $512 — $200 short — because the "
    "$250 deductible was wrongly re-applied. Decision: "
    "approve_with_adjustment — refund the $200 difference.'\n"
    "\n"
    "WRONG vs RIGHT — observed failure mode:\n"
    "  WRONG: 'Your issue has been escalated to a human reviewer "
    "due to insufficient information.' (no Section, no query terms, "
    "no reason — violates RULES 1 + 3)\n"
    "  RIGHT: 'Per Section 4.7 of <policy_id>, ...' OR an escalation "
    "that names what was searched and which sections were inadequate.\n"
)

In [0]:
llm = ChatDatabricks(endpoint=llm_endpoint)

# Source: https://langchain-ai.github.io/langgraph/reference/agents/
# Current `create_react_agent` signature (langgraph >= 0.3) takes `prompt=`;
# the older `state_modifier=` was deprecated and removed.
agent_graph = create_react_agent(
    llm,
    tools=all_tools,
    prompt=SYSTEM_PROMPT,
)

/home/spark-b2431b4d-1564-4f48-9446-b1/.ipykernel/66792/command-7841716993424945-604226987:6: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_graph = create_react_agent(


## 6. Wrap in ResponsesAgent for deployment compatibility

In [0]:
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import ResponsesAgentRequest, ResponsesAgentResponse

import uuid

class ClaimClerkResponsesAgent(ResponsesAgent):
    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        # Source: https://mlflow.org/docs/latest/genai/serving/responses-agent
        # `request.input` is a list of mlflow Message objects (Responses-API
        # shape) — convert to LangChain-compatible role/content dicts.
        messages = [
            {"role": m.role, "content": m.content}
            for m in request.input
        ]
        # `request.custom_inputs` is the documented per-request scope
        # channel. Use it for caller-side facts (customer_id, tenant_id,
        # language preference) that the agent should treat as context
        # rather than parsing them out of the user message text.
        custom = getattr(request, "custom_inputs", None) or {}
        customer_id = custom.get("customer_id")
        if customer_id:
            messages = [{
                "role": "system",
                "content": f"The authenticated caller is customer_id={customer_id}.",
            }] + messages

        result = agent_graph.invoke({"messages": messages})
        final = result["messages"][-1].content
        # Responses-API output items require `id` and a typed `content` list.
        return ResponsesAgentResponse(
            output=[{
                "type": "message",
                "id": str(uuid.uuid4()),
                "role": "assistant",
                "content": [{"type": "output_text", "text": final}],
            }]
        )

clerk = ClaimClerkResponsesAgent()

## 7. Demo — Sarah Chen's appeal

The lifecycle anchor. The agent should: (1) look up Sarah's
policies, find Biscuit's PawPlus coverage; (2) retrieve the
deductible-handling clause (Section 4.7) for TX PawPlus from
the policy_chunks_index via MCP; (3) reason that the
deductible was already satisfied earlier in the year by claim
CLM-2026-01-00098 and shouldn't recur; (4) produce an answer
citing Section 4.7.

In [0]:
sarah_question = (
    "I submitted Biscuit's ER claim two weeks ago (CLM-2026-03-00471) "
    "and only got $512 back on an $890 bill. I'm being told it's because "
    "of my deductible but I already paid that this year. Can you explain "
    "what happened?"
)

In [0]:
# Pass customer_id via `custom_inputs`, NOT embedded in the message text.
# The authenticated caller's identity is per-request scope: the App layer
# knows who the user is and should pass that as data, not let the LLM
# parse it out of free text. custom_inputs is the documented per-request-scope contract.
response = clerk.predict(ResponsesAgentRequest(
    input=[{"role": "user", "content": sarah_question}],
    custom_inputs={"customer_id": "CUST-CHEN-001"},
))
# The new output shape is `output[0].content = [{type: output_text, text: ...}]`.
first = response.output[0]
content = first["content"] if isinstance(first, dict) else first.content
print(content[0]["text"] if isinstance(content[0], dict) else content[0].text)

/local_disk0/.ephemeral_nfs/envs/pythonEnv-b2431b4d-1564-4f48-9446-b1c7e96109f6/lib/python3.10/site-packages/unitycatalog/ai/core/databricks.py:594: UserWarning: The following parameters do not have descriptions: customer_id for the function genaicert.pawshield.get_customer_policy. Using Unity Catalog functions that do not have parameter descriptions limits the functionality for an LLM to understand how to call your function. To improve tool calling accuracy, provide verbose parameter descriptions that fully explain what the expected usage of the function arguments are.
  check_function_info(function_info)
/local_disk0/.ephemeral_nfs/envs/pythonEnv-b2431b4d-1564-4f48-9446-b1c7e96109f6/lib/python3.10/site-packages/unitycatalog/ai/core/databricks.py:594: UserWarning: The following parameters do not have descriptions: customer_id, policy_year for the function genaicert.pawshield.get_claim_history. Using Unity Catalog functions that do not have parameter descriptions limits the functionali

Per Section 4.7 of policy POL-2025-01-CHENB02, the $250 annual deductible applies once per policy year and resets at renewal. Your January claim ($420 billed) had the $250 deductible applied, then 80% coinsurance on the remaining $170 = $136 reimbursed — correct. By March, the deductible was already satisfied, so it should not apply again: 80% of the $890 bill = $712 owed. You were reimbursed only $512 — $200 short — because the $250 deductible was wrongly re-applied. Decision: approve_with_adjustment — refund the $200 difference.


Trace(trace_id=tr-d773d9dbc21142b023dc356ba393c750)

## 8. Inspect the trace

The agent run produces an MLflow trace with one span per tool
call. Open the most recent run in MLflow UI to see:

* The order tools were called (the reasoning loop)
* What each tool returned (UC function rows, VS MCP chunks)
* The final LLM call that composed the answer

Open the MLflow Tracing UI on the predict run to inspect the
trace view.

## Register the agent for code-based logging

`mlflow.models.set_model(...)` exposes the agent instance to
`mlflow.pyfunc.log_model(python_model="c0901-claimclerk-agent-with-mcp.py")`
for deployment. The same file becomes both the in-notebook demo (the
cells above) and the deployable artifact (the serving
pipeline) — no separate `agent.py` needed.
Source: https://docs.databricks.com/aws/en/generative-ai/agent-framework/log-agent

In [0]:
import mlflow
mlflow.models.set_model(clerk)

## What's next

* The canonical deployment notebook `c1001-deploy-policypal.py`
  ships **PolicyPal** — the same `mlflow.pyfunc.log_model(...)` +
  Model Serving wrapping applies to ClaimClerk's agent endpoint
  when a team decides to deploy it live. The `ClaimClerkResponsesAgent`
  built in this notebook is the artifact you'd hand to that
  same pattern, with ClaimClerk's UC-function tools declared
  in `resources=[...]` so auto-auth grants the served SP
  `EXECUTE` on each function.
* PII redaction + jailbreak detection: for ClaimClerk's
  custom-PyFunc agent endpoint, the AI Gateway guardrails surface
  is *not* available — gateway guardrails apply only to External
  Models + FM-API pay-per-token endpoints. Custom agents enforce
  in-chain (preprocess input / postprocess output inside predict).
* The `c1301` evaluation notebook evaluates the agent end-to-end
  with `mlflow.genai.evaluate` and custom Scorers.